In [16]:
import pandas as pd
import numpy as np
import dblp
from crossref.restful import Works
import requests

In [8]:
crossref_work = Works()

In [68]:
raw_df = pd.read_csv('data/search-results.csv', index_col=0)

In [69]:
# Remove duplicates
raw_df = raw_df.drop_duplicates(subset=['PaperTitle', 'DOI'])

In [5]:
# Prepare DataFrame to store the results
columns = [
    "PaperTitle",
    "DOI",
    "Authors",
    "Abstract",
    "Publisher",
    "DoiUrl",
    "PublicationDate",
    "Conference-Journal",
    "PublicationTypes",
    "SearchString",
    "CitationCount",
    "SearchedFrom",
]

In [39]:
fields = [
    "title",
    "externalIds",
    "authors",
    "abstract",
    "url",
    "publicationDate",
    "fieldsOfStudy",
    "venue",
    "publicationTypes",
    "citationCount",
    "externalIds",
]

In [63]:
def extract_data(sch_paper, row):
        doi = sch_paper.get("externalIds", {}).get("DOI", None)
        if doi is None and str(row["DOI"]).startswith("DOI"):
            doi = row["DOI"].split(":")[1]
        try:
            crossref_paper = crossref_work.doi(doi)
        except Exception as e:
            crossref_paper = None

        title = row["PaperTitle"]

        authors = None
        # author name and affiliation
        if crossref_paper is not None:
            authors = crossref_paper.get("author")
        if authors is not None:
            for i in range(len(authors)):
                author = authors[i]
                author_name = author.get("given", "") + " " + author.get("family", "")
                affiliation = author.get("affiliation", "No Affiliation")
                affiliations = author.get("affiliation", [])
                school_names = (
                    [affil.get("name") for affil in affiliations]
                    if affiliations
                    else ["No Affiliation"]
                )
                # Create a new dictionary with only 'name' and 'affiliation'
                authors[i] = {
                    "name": author_name.strip(),
                    "affiliation": school_names,
                }
        else:
            authors = sch_paper.get("authors", [])
            for i in range(len(authors)):
                author = sch_paper["authors"][i]
                sch_paper["authors"][i] = {
                    "name": author.get("name", "No Name"),
                    "affiliation": author.get("affiliation", "No Affiliation"),
                }

        abstract = sch_paper.get("abstract", None)
        sch_url = sch_paper.get("url", None)
        doi_url = f"https://doi.org/{doi}"
        publication_date = sch_paper.get("publicationDate", None)
        fields_of_study = sch_paper.get("fieldsOfStudy", [])
        venue = sch_paper.get("venue", None)

        # publisher
        if crossref_paper is not None:
            publisher = crossref_paper.get("publisher")
        elif doi and "arxiv" in doi.lower():
            publisher = "arXiv"
        else:
            publisher = None

        # paper type
        if crossref_paper is not None:
            paper_type = [crossref_paper.get("type")]
        else:
            paper_type = sch_paper.get("publicationTypes", [])

        citation_count = sch_paper.get("citationCount", None)
        # TODO: paper keywords missing
        # TODO: paper type is conference/journal for arxiv papers
        # TODO: conference-journal name mismatch with publisher, i.e., for paper with name"ChatGPT in education: A discourse analysis of worries and concerns on social media", the conference name is "International Conference on Artificial Intelligence in Education", but the publisher is "Arxiv" (becauseit queryed from arxiv), need "Springer" instead.

        new_paper = {
            "PaperTitle": title,
            "DOI": doi,
            "Authors": authors,
            "Abstract": abstract,
            "Publisher": publisher,
            "SemanticScholarUrl": sch_url,
            "DoiUrl": doi_url,
            "PublicationDate": publication_date,
            "FieldOfStudy": fields_of_study,
            "Conference-Journal": venue,
            "PublicationTypes": paper_type,
            "SearchString": row["SearchString"],
            "CitationCount": citation_count,
            "SearchedFrom": row["SearchedFrom"],
        }
        return new_paper

In [70]:
# loop through rows

results = []

for index, row in raw_df.iterrows():
    # send http request to api
    api_url = f"https://api.semanticscholar.org/graph/v1/paper/{row['DOI']}"
    headers = {"Content-Type": "application/json"}
    params = {
        "fields": ",".join(fields),
    }
    response = requests.get(api_url, headers=headers, params=params)
    result = response.json()
    paper = extract_data(result, row)
    results.append(paper)

In [72]:
results_df = pd.DataFrame(results, columns=columns)
results_df.to_csv("data/initial-scrape-result.csv", index=False)